In [2]:
import pandas as pd

In [ ]:
# Load and view
df =pd. read_csv('../data/Walmart.csv')
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [ ]:
# Convert the data column into datetime format
df['Date'] = pd.to_datetime(df['Date'], format="%d-%m-%Y")
df

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106
...,...,...,...,...,...,...,...,...
6430,45,2012-09-28,713173.95,0,64.88,3.997,192.013558,8.684
6431,45,2012-10-05,733455.07,0,64.89,3.985,192.170412,8.667
6432,45,2012-10-12,734464.36,0,54.47,4.000,192.327265,8.667
6433,45,2012-10-19,718125.53,0,56.47,3.969,192.330854,8.667


In [ ]:
# Sort the dataset by date and reset index
df = df.sort_values(by ='Date', ascending=True).reset_index(drop=True)


In [ ]:
# Set column titles to lowercase
df.columns = df.columns.str.lower()
df

,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,10,2010-02-05,2193048.75,0,54.34,2.962,126.442065,9.765
2,37,2010-02-05,536006.73,0,45.97,2.572,209.852966,8.554
3,17,2010-02-05,789036.02,0,23.11,2.666,126.442065,6.548
4,30,2010-02-05,465108.52,0,39.05,2.572,210.752605,8.324
...,...,...,...,...,...,...,...,...
6430,25,2012-10-26,688940.94,0,56.69,3.882,216.151590,7.293
6431,5,2012-10-26,319550.77,0,71.70,3.506,224.037814,5.422
6432,40,2012-10-26,921264.52,0,49.65,3.917,138.728161,4.145
6433,18,2012-10-26,1127516.25,0,56.09,3.917,138.728161,8.243


In [16]:
# Check for duplicates
duplicates = df[df.duplicated(keep=False)]
duplicates


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment


In [ ]:
# Check for null values
print(df.isnull().sum())


store           0
date            0
weekly_sales    0
holiday_flag    0
temperature     0
fuel_price      0
cpi             0
unemployment    0
dtype: int64


In [ ]:
# check for negative numbers
df_clean = df.copy()
cols_to_convert = df_clean.columns.drop('date')
df_clean[cols_to_convert] = df_clean[cols_to_convert].apply(pd.to_numeric)
print((df_clean.select_dtypes(include='number')< 0).sum())

store           0
weekly_sales    0
holiday_flag    0
temperature     1
fuel_price      0
cpi             0
unemployment    0
dtype: int64


In [15]:
# Check for invalid holiday
invalid_holiday = df_clean[~df_clean['holiday_flag'].isin([0,1])]
print(len(invalid_holiday))

0


In [17]:
# Check for invalid fuel prices
invalid_fuel = df_clean[df_clean['fuel_price'] <= 0]
print(len(invalid_fuel))

0


In [ ]:
 # Create a report directory  -- previously handled  in src/data.py
import os

os.makedirs('reports')

# Build data-quaity summary metric
dq_metrics = []

for col in df_clean.columns:
    total_rows = len(df)
    missing_count = df_clean[col].isnull().sum()
    dtype = str(df_clean[col].dtype) # except datetime

# Check if column is numeric to calculate stats and outliers
if pd.api.types.is_numeric_dtype(df[col]):
    min_val = df_clean[col].min()
    max_val = df_clean[col].max()

    # Outlier detection using IQR
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

else:
    min_val = df[col].min() if col == 'date' else 'N/A'
    max_val = df[col].max() if col == 'date' else 'N/A'
    outlier_count = 'N/A'

# Annotate values
dq_metrics.append({
    'Column Name': col,
      'Data Type': dtype,
      'Total Rows': total_rows,
      'Missing Count': missing_count,
      'Min Value': min_val,
      'Max Value': max_val,
      'Outliers Flagged (IQR)': outlier_count,
})

dq_df =pd.DataFrame(dq_metrics)

# Save to reports
output_path = 'reports/data_quality_report.csv'
dq_df.to_csv(output_path, index=False)
